# 🐍 Day 4: functools

- reduce for folding sequences
- partial for partial application
- lru_cache for memoization
- wraps, total_ordering, singledispatch

## reduce

`reduce(function, iterable, initializer)` applies a binary function cumulatively: f(f(f(init, x1), x2), x3)...

In [ ]:
from functools import reduce

total = reduce(lambda a, b: a + b, [1, 2, 3, 4, 5])
print(total)

product = reduce(lambda a, b: a * b, [1, 2, 3, 4, 5])
print(product)

# With initializer
result = reduce(lambda a, b: a + b, [1, 2, 3], 10)
print(result)  # 10+1+2+3 = 16

## partial - Partial Application

`partial(func, *args, **kwargs)` creates a new callable with some arguments pre-filled.

In [ ]:
from functools import partial

def power(base, exp):
    return base ** exp

square = partial(power, exp=2)
cube = partial(power, exp=3)
print(square(5))
print(cube(5))

# Pre-fill first arg
double = partial(lambda x, m: x * m, 2)
print(double(10))  # 2 * 10? No - 10 is second arg. Use:
double = lambda x: x * 2  # Or partial with keyword
from operator import mul
double = partial(mul, 2)
print(double(10))

## lru_cache - Memoization

`@lru_cache(maxsize=128)` caches function results. Same args → return cached value. Great for recursive functions like Fibonacci.

In [ ]:
from functools import lru_cache

@lru_cache(maxsize=None)
def fib(n):
    if n < 2:
        return n
    return fib(n - 1) + fib(n - 2)

print(fib(35))
print(fib.cache_info())

## wraps - Preserve Metadata

`@wraps(original_func)` copies __name__, __doc__, etc. to the wrapper. Essential for decorators.

In [ ]:
from functools import wraps

def my_decorator(f):
    @wraps(f)
    def wrapper(*args, **kwargs):
        print("Before")
        result = f(*args, **kwargs)
        print("After")
        return result
    return wrapper

@my_decorator
def say_hello():
    """Says hello."""
    print("Hello!")

say_hello()
print(say_hello.__name__)
print(say_hello.__doc__)

## total_ordering - Minimal Comparisons

`@total_ordering` generates __le__, __gt__, __ge__ from __eq__ and one of __lt__, __le__, __gt__, __ge__.

In [ ]:
from functools import total_ordering

@total_ordering
class Student:
    def __init__(self, name, grade):
        self.name = name
        self.grade = grade

    def __eq__(self, other):
        if not isinstance(other, Student):
            return NotImplemented
        return self.grade == other.grade

    def __lt__(self, other):
        if not isinstance(other, Student):
            return NotImplemented
        return self.grade < other.grade

s1 = Student("Alice", 85)
s2 = Student("Bob", 90)
print(s1 < s2)
print(s1 >= s2)

## singledispatch - Function Overloading

`@singledispatch` creates a generic function with type-based dispatch. Register implementations for specific types.

In [ ]:
from functools import singledispatch

@singledispatch
def process(arg):
    return f"default: {arg}"

@process.register(int)
def _(arg):
    return f"int: {arg * 2}"

@process.register(list)
def _(arg):
    return f"list: {len(arg)} items"

print(process(10))
print(process([1, 2, 3]))
print(process("hello"))

## cached_property (Python 3.8+)

`@cached_property` computes once and caches. Like property but lazy and cached.

In [ ]:
from functools import cached_property

class Data:
    def __init__(self, items):
        self.items = items

    @cached_property
    def total(self):
        print("Computing total...")
        return sum(self.items)

d = Data([1, 2, 3, 4, 5])
print(d.total)
print(d.total)  # cached, no second print

## Practical: Composable Pipeline

Use reduce to chain functions.

In [ ]:
from functools import reduce

def pipeline(data, *functions):
    return reduce(lambda x, f: f(x), functions, data)

result = pipeline(
    "  hello  ",
    str.strip,
    str.upper,
    lambda s: s + "!"
)
print(result)

## Common Mistakes

- **reduce with no initializer**: For empty iterable, raises TypeError
- **lru_cache with unhashable args**: Args must be hashable (no lists/dicts)
- **Forgetting @wraps**: Wrapper loses original function's name/doc
- **total_ordering**: Must implement __eq__ and one comparison (__lt__ recommended)

In [ ]:
# WRONG: lru_cache with list arg - unhashable
# @lru_cache()
# def process(items): return sum(items)
# process([1,2,3])  # TypeError: unhashable type: 'list'

# CORRECT: Use tuple (hashable)
from functools import lru_cache
@lru_cache()
def process_tuple(items_tuple):
    return sum(items_tuple)
print(process_tuple((1, 2, 3)))

## Exercises

In [ ]:
# Exercise 1: Use reduce to find the maximum of [3, 1, 4, 1, 5].
# YOUR CODE HERE

In [ ]:
# Exercise 2: Use partial to create a function that multiplies by 10. Call it with 7.
# YOUR CODE HERE

In [ ]:
# Exercise 3: Add @lru_cache to a factorial function. Compute factorial(10) twice and check cache_info.
# YOUR CODE HERE

In [ ]:
# Exercise 4: Create a decorator that prints function name. Use @wraps. Verify __name__ is preserved.
# YOUR CODE HERE

In [ ]:
# Exercise 5: Use @total_ordering with __eq__ and __lt__ on a simple class. Compare two instances.
# YOUR CODE HERE

In [ ]:
# Exercise 6: Use singledispatch to handle int (double it) and str (uppercase). Test both.
# YOUR CODE HERE

## Key Takeaways

1. `reduce(f, it, init)` folds sequence with binary function
2. `partial(f, *a, **k)` pre-fills arguments
3. `@lru_cache` memoizes; args must be hashable
4. `@wraps(f)` preserves metadata in decorators
5. `@total_ordering` needs __eq__ + one comparison
6. `@singledispatch` for type-based overloading

## 🔜 Next: Day 5 - Builtins Advanced

Tomorrow: map, filter, zip, enumerate, any, all, sorted with key, reversed!